In [2]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8


In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key="aDBcizPBbowCwQqldqlk")
project = rf.workspace("ayushs-workspace-l4nr5").project("shelfsense-yhthy")
version = project.version(2)
dataset = version.download("yolov8")

print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...
/content/ShelfSense-2


In [5]:
!cat {dataset.location}/data.yaml

names:
- Barcode
- box
- pallet
- shelf
nc: 4
roboflow:
  license: CC BY 4.0
  project: shelfsense-yhthy
  url: https://universe.roboflow.com/ayushs-workspace-l4nr5/shelfsense-yhthy/dataset/2
  version: 2
  workspace: ayushs-workspace-l4nr5
test: ../test/images
train: ../train/images
val: ../valid/images


In [6]:
# Remove barcode from the dataset

import glob

DATASET = dataset.location
DROP_ID = 0

files, removed = 0, 0
for split in ["train", "valid", "test"]:
    for path in glob.glob(f"{DATASET}/{split}/labels/*.txt"):
        kept = []
        for line in open(path).read().splitlines():
            parts = line.split()
            if not parts:
                continue
            cid = int(parts[0])
            if cid == DROP_ID:
                removed += 1
                continue
            if cid > DROP_ID:
                parts[0] = str(cid - 1)
            kept.append(" ".join(parts))
        open(path, "w").write("\n".join(kept))
        files += 1

print(f"processed {files} label files · evicted {removed} Barcode boxes")

processed 1707 label files · evicted 551 Barcode boxes


In [7]:
# Update the data.yaml without the barcode

import yaml
p = f"{DATASET}/data.yaml"
d = yaml.safe_load(open(p))
d["names"] = ["box", "pallet", "shelf"]
d["nc"] = 3
yaml.safe_dump(d, open(p, "w"))
print(open(p).read())

names:
- box
- pallet
- shelf
nc: 3
roboflow:
  license: CC BY 4.0
  project: shelfsense-yhthy
  url: https://universe.roboflow.com/ayushs-workspace-l4nr5/shelfsense-yhthy/dataset/2
  version: 2
  workspace: ayushs-workspace-l4nr5
test: ../test/images
train: ../train/images
val: ../valid/images



In [9]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

results = model.train(
    data = f"{DATASET}/data.yaml",
    epochs = 100,
    imgsz = 640,
    batch = 16,
    patience = 20,
)

Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ShelfSense-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [10]:
from google.colab import files
files.download("runs/detect/train/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>